In [1]:
import sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV

sys.path.append(r"C:\Users\arbaz2\Desktop\Quant Finance\Volatility Forecasting\src")

from data_pipeline import make_dataset
from baselines import make_baseline_forecasts
from metrics import qlike, mse, score, score_by_regime, score_by_ticker
from garch import add_garch_refit_recurse, add_gjr_garch_forecast, plot_vol_compare
from har_rv_ensemble import add_har_forecasts

CRISIS_WINDOWS = {
    "GFC_2007_2009": ("2007-07-01", "2009-06-30"),
    "COVID_2020": ("2020-02-15", "2020-05-31"),
}



In [2]:
df = make_dataset(["SPY", "JPM"], start="2000-01-01", horizons=(1,5), crisis_windows=CRISIS_WINDOWS)
df["date"] = pd.to_datetime(df["date"])

In [3]:
df = make_baseline_forecasts(df, hv_window=20, ewma_lam=0.94)
df.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.490305,4723500,-1.733118,3.003698,5.704780,14.393831,calm,NaN,NaN,5.320387,26.601937
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.877930,5741700,0.342438,0.117264,1.449174,36.092014,calm,NaN,NaN,1.474746,7.373732
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.959496,8405550,-2.388468,5.704780,0.390288,14.400641,calm,NaN,NaN,5.181386,25.906930
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.778519,7503700,-1.203817,1.449174,0.999418,37.059499,calm,NaN,NaN,1.393297,6.966487
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,22.097113,7271850,0.624731,0.390288,2.253480,14.666743,calm,NaN,NaN,5.212790,26.063948


In [5]:
df = add_garch_refit_recurse(df, refit_every=5, min_train=750, mean="zero", dist="t")
df.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var,garch1_var,garch5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.605406,4723500,-1.733151,3.003814,5.704727,14.393769,calm,NaN,NaN,5.354318,26.771592,NaN,NaN
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,92.128860,5741700,0.342468,0.117284,1.448981,36.091913,calm,NaN,NaN,1.480350,7.401750,NaN,NaN
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,22.071882,8405550,-2.388457,5.704727,0.390361,14.400761,calm,NaN,NaN,5.213288,26.066441,NaN,NaN
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,91.026520,7503700,-1.203736,1.448981,0.999649,37.059608,calm,NaN,NaN,1.398566,6.992830,NaN,NaN
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,22.210217,7271850,0.624788,0.390361,2.253152,14.666895,calm,NaN,NaN,5.242775,26.213873,NaN,NaN


In [6]:
df = add_gjr_garch_forecast(df, refit_every=21, min_train=750, dist="t" )
df.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,...,rv5_var,regime,hv1_var,hv5_var,ewma1_var,ewma5_var,garch1_var,garch5_var,gjr1_var,gjr5_var
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.605406,4723500,-1.733151,3.003814,...,14.393769,calm,NaN,NaN,5.354318,26.771592,NaN,NaN,NaN,NaN
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,92.128860,5741700,0.342468,0.117284,...,36.091913,calm,NaN,NaN,1.480350,7.401750,NaN,NaN,NaN,NaN
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,22.071882,8405550,-2.388457,5.704727,...,14.400761,calm,NaN,NaN,5.213288,26.066441,NaN,NaN,NaN,NaN
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,91.026520,7503700,-1.203736,1.448981,...,37.059608,calm,NaN,NaN,1.398566,6.992830,NaN,NaN,NaN,NaN
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,22.210217,7271850,0.624788,0.390361,...,14.666895,calm,NaN,NaN,5.242775,26.213873,NaN,NaN,NaN,NaN


In [7]:
df_har = add_har_forecasts(df, refit_every=21, min_train=252)

In [26]:
def make_enet_features_single(d):
    """
    Create Elastic Net features for one ticker.

    The features include:
    - HAR-style realized variance features
    - lagged returns
    - lagged squared returns
    """
    d = d.sort_values("date").copy()

    # squared returns
    d["ret2"] = d["ret"] ** 2

    # HAR-style features from realized variance
    d["rv_d"] = d["rv1_var"].shift(1)                 # yesterday's realized variance
    d["rv_w"] = d["rv1_var"].shift(1).rolling(5).mean()   # last 5-day average
    d["rv_m"] = d["rv1_var"].shift(1).rolling(22).mean()  # last 22-day average

    # lagged returns
    d["ret_l1"] = d["ret"].shift(1)
    d["ret_l2"] = d["ret"].shift(2)
    d["ret_l5_mean"] = d["ret"].shift(1).rolling(5).mean()

    # lagged squared returns
    d["ret2_l1"] = d["ret2"].shift(1)
    d["ret2_l5_mean"] = d["ret2"].shift(1).rolling(5).mean()
    d["ret2_l22_mean"] = d["ret2"].shift(1).rolling(22).mean()

    return d

In [27]:
def add_enet_features(df):
    """
    Apply Elastic Net feature construction separately to each ticker.
    """
    out = df.copy().sort_values(["ticker", "date"]).reset_index(drop=True)

    pieces = []
    for tkr in out["ticker"].unique():
        d = out[out["ticker"] == tkr].copy()
        d = make_enet_features_single(d)
        pieces.append(d)

    # combine all tickers back into one panel dataframe
    out = pd.concat(pieces, axis=0).sort_values(["date", "ticker"]).reset_index(drop=True)
    return out

In [35]:
def enet_rolling_forecast_single(
    d,
    target_col="rv1_var",
    refit_every=21,
    min_train=252
):
    """
    Expanding-window Elastic Net forecast for one ticker.

    Important:
    - We fit Elastic Net to log(variance) instead of variance directly.
    - This makes the target more stable and ensures positive forecasts after exponentiating.
    """
    d = d.sort_values("date").copy()

    feature_cols = [
        "rv_d", "rv_w", "rv_m",
        "ret_l1", "ret_l2", "ret_l5_mean",
        "ret2_l1", "ret2_l5_mean", "ret2_l22_mean"
    ]

    # container for forecasts
    fcast = np.full(len(d), np.nan)

    # start forecasting only after minimum training history
    t = min_train

    while t < len(d):
        # training sample up to time t
        train = d.iloc[:t+1].dropna(subset=feature_cols + [target_col]).copy()

        if len(train) < min_train:
            t += refit_every
            continue

        # feature matrix
        X_train = train[feature_cols].values

        # IMPORTANT FIX:
        # fit the model on log variance instead of raw variance
        # log1p stabilizes variance and avoids extreme values
        y_train = np.log1p(train[target_col].values)

        # pipeline:
        # 1) standardize predictors
        # 2) fit Elastic Net with CV over alpha and l1_ratio
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("enet", ElasticNetCV(
                l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                alphas=np.logspace(-4, 1, 30),
                cv=5,
                max_iter=10000
            ))
        ])

        # fit on training window
        model.fit(X_train, y_train)

        # forecast until next refit
        t_end = min(t + refit_every, len(d))

        for i in range(t, t_end):
            row = d.iloc[i]

            # skip if any feature is missing
            if row[feature_cols].isna().any():
                continue

            X_test = row[feature_cols].values.reshape(1, -1)

            # predict log variance
            pred_log = model.predict(X_test)[0]

            # invert log1p transformation
            pred = np.expm1(pred_log)

            #ensure the it is positive
            pred = max(pred, 1e-6)

            fcast[i] = pred

        t = t_end

    return pd.Series(fcast, index=d.index)

In [36]:
def add_enet_forecasts(
    df,
    refit_every=21,
    min_train=252
):
    """
    Add Elastic Net forecasts for both 1-day and 5-day horizons.

    Adds:
    - enet1_var : forecast for rv1_var
    - enet5_var : forecast for rv5_var
    """
    # first create all the features
    out = add_enet_features(df)
    out = out.sort_values(["ticker", "date"]).reset_index(drop=True)

    enet1_list = []
    enet5_list = []

    for tkr in out["ticker"].unique():
        d = out[out["ticker"] == tkr].copy()

        # 1-day forecast
        s1 = enet_rolling_forecast_single(
            d,
            target_col="rv1_var",
            refit_every=refit_every,
            min_train=min_train
        )

        # 5-day forecast
        s5 = enet_rolling_forecast_single(
            d,
            target_col="rv5_var",
            refit_every=refit_every,
            min_train=min_train
        )

        enet1_list.append(s1.rename(tkr))
        enet5_list.append(s5.rename(tkr))

    # combine all ticker forecasts
    enet1_all = pd.concat(enet1_list, axis=0).sort_index()
    enet5_all = pd.concat(enet5_list, axis=0).sort_index()

    # add to dataframe
    out["enet1_var"] = enet1_all
    out["enet5_var"] = enet5_all

    return out.sort_values(["date", "ticker"]).reset_index(drop=True)

In [37]:
df_enet = add_enet_forecasts(df_har, refit_every=21, min_train=252)

In [38]:
models_1d = ["hv1_var", "ewma1_var", "garch1_var", "gjr1_var", "har1_var", "enet1_var"]
print(score(df_enet, models_1d, "rv1_var", eval_start="2005-01-01"))
print()

models_5d = ["hv5_var", "ewma5_var", "garch5_var", "gjr5_var", "har5_var", "enet5_var"]
print(score(df_enet, models_5d, "rv5_var", eval_start="2005-01-01"))

print(score_by_regime(df_enet, models_1d, "rv1_var", eval_start="2005-01-01"))

        model   target      n     QLIKE         MSE
3    gjr1_var  rv1_var  10652  1.359029  213.055396
2  garch1_var  rv1_var  10652  1.393298  220.019759
4    har1_var  rv1_var  10652  1.446941  221.665824
1   ewma1_var  rv1_var  10652  1.450793  226.174362
0     hv1_var  rv1_var  10652  1.490455  235.490012
5   enet1_var  rv1_var  10652  1.639691  325.512646

        model   target      n     QLIKE           MSE
4    har5_var  rv5_var  10652  2.730367  3.755649e+02
3    gjr5_var  rv5_var  10652  2.831692  9.998816e+02
2  garch5_var  rv5_var  10652  2.844597  1.058161e+03
0     hv5_var  rv5_var  10652  2.890883  1.456067e+03
1   ewma5_var  rv5_var  10652  2.911795  1.330491e+03
5   enet5_var  rv5_var  10652  3.045368  1.428783e+16
           regime       model   target     n     QLIKE          MSE
3      COVID_2020    gjr1_var  rv1_var   144  3.724733  1546.464726
4      COVID_2020    har1_var  rv1_var   144  3.935394  1460.545814
2      COVID_2020  garch1_var  rv1_var   144  3.96835

In [33]:
import plotly.express as px
px.histogram(df_enet, x="enet1_var", nbins=100, title="Elastic Net Forecast Distribution")

In [34]:
df_enet["enet1_var"].describe()

count    1.260800e+04
mean     2.624984e+03
std      2.196718e+05
min      9.233045e-02
25%      1.930584e-01
50%      4.501685e-01
75%      6.042171e-01
max      2.241859e+07
Name: enet1_var, dtype: float64

In [25]:
plot_vol_compare(
    df_enet,
    "SPY",
    crisis_windows = CRISIS_WINDOWS,
    target_var="rv1_var",
    forecast_vars=("gjr1_var", "har1_var", "enet1_var"),
    start="2020-02-15",
    end="2020-05-31",
    name = "COVID",
    title="SPY: GJR vs HAR vs Elastic Net (1-day)",
    
)

In [15]:
plot_vol_compare(
    df_enet,
    "SPY",
    target_var="rv5_var",
    forecast_vars=("gjr5_var", "har5_var", "enet5_var"),
    start="2020-02-15",
    end="2020-05-31",
    name = "COVID",
    title="SPY: GJR vs HAR vs Elastic Net (5-day)",
    crisis_windows = CRISIS_WINDOWS
)

### I explored a simple Machine Learning benchmark, but structured volatility models remained more robust/ and performant.

Issues with ElasticNet
* it requires positivity/log-targeting handling
* it produced unstable long-horizon behavior
* it did not outperform GJR/HAR